# K-TIDE — Official CUDA Baseline Runner (Colab)

Runs inference with the **professor's own `load_model` / `answer_questions`** (parsed from the private notebook at runtime) under the official conditions: CUDA FP16, SDPA, greedy, batch 4, left padding.

1. Runtime → GPU (T4 is fine for FP16).
2. Run the cells in order. You will be asked to upload the professor's notebook (`TEST_Qwen25_SFT.ipynb`). It stays in this Colab VM only.
3. Download the resulting run ZIP and put it under `runs/` locally, then run the judge locally.

This notebook contains no private material and is safe to keep in the public repo.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/Hun-Bot2/K-TIDE.git"  # or upload the repo and set REPO_DIR
REPO_DIR = "/content/K-TIDE"
GIT_REF = "main"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
# Same pinned versions as the professor's notebook; torch = Colab preinstalled (recorded in environment.json).
os.environ["USE_TF"] = "0"; os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"; os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
    "transformers==4.57.1", "accelerate==1.11.0", "huggingface_hub==0.36.0",
    "safetensors==0.6.2", "gdown==5.2.0", "pydantic>=2.9"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_DIR], check=True)

In [ ]:
from google.colab import files
dest = f"{REPO_DIR}/data_private/professor"
os.makedirs(dest, exist_ok=True)
if not os.path.isfile(f"{dest}/TEST_Qwen25_SFT.ipynb"):
    up = files.upload()  # select TEST_Qwen25_SFT.ipynb
    name = next(iter(up))
    os.replace(name, f"{dest}/TEST_Qwen25_SFT.ipynb")
os.environ["KTIDE_PROFESSOR_NOTEBOOK"] = f"{dest}/TEST_Qwen25_SFT.ipynb"

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.__version__, torch.cuda.get_device_name(0))
r = subprocess.run([sys.executable, "-m", "ktide.generate", "--experiment", "baseline",
                    "--condition", "B0", "--tag", "colab-official",
                    "--notes", "Official conditions: Colab CUDA FP16 via professor load_model."],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])

In [ ]:
import glob, shutil, json
run_dir = sorted(glob.glob(f"{REPO_DIR}/runs/baseline/B0/*colab-official*"))[-1]
print(open(f"{run_dir}/judge_input.txt", encoding="utf-8").read())
print(json.dumps(json.load(open(f"{run_dir}/config.json"))["inference"]["generation_config"], indent=2))
archive = shutil.make_archive(f"/content/{os.path.basename(run_dir)}", "zip", run_dir)
files.download(archive)